## Exploring Presidio for PII discovery and pseudonymization

[Microsoft Presidio](https://microsoft.github.io/presidio/) uses natural language processing software ([SpaCy](https://spacy.io/)) to assist in finding PII and pseudonymizing or redacting it.

In this notebook, you'll explore how to use it, where to use it and also when to not use it.

In [ ]:
!python -m spacy download en_core_web_lg

In [ ]:
!pip install presidio_analyzer presidio_anonymizer

You should also install [tesseract](https://tesseract-ocr.github.io/tessdoc/Installation.html) if you want to use it with images.

In [ ]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import os

In [ ]:
text_to_pseudonymize = "His name is Mr. Samuel Jones and his phone number is +49 110 0342121345"

In [ ]:
analyzer = AnalyzerEngine()
analyzer_results = analyzer.analyze(text=text_to_pseudonymize, 
                                    entities=["PHONE_NUMBER", "PERSON"], 
                                    language='en')


In [ ]:
analyzer_results

A few more entities are [found in the full list](https://microsoft.github.io/presidio/supported_entities/).

In [ ]:
supported_entities = [

    "PHONE_NUMBER",
    "US_DRIVER_LICENSE",
    "US_PASSPORT",
    "LOCATION",
    "CREDIT_CARD",
    "CRYPTO",
    "UK_NHS",
    "US_SSN",
    "US_BANK_NUMBER",
    "EMAIL_ADDRESS",
    "DATE_TIME",
    "IP_ADDRESS",
    "PERSON",
    "IBAN_CODE",
    "NRP",
    "US_ITIN",
    "MEDICAL_LICENSE",
    "URL"

]


## Practice: 
- Write a few more sentences and test out some of the other entities.
- You can also change the language. You might need to install additional packages, so [check out the SpaCy documentation](https://spacy.io/usage/models/#languages).

In [ ]:
more_text = """
Yo, can you please call me back -- you have my number under JJ already. 
It's urgent because your dad is in the hospital and they are asking for his identity details. 
I htink his SS number starts with 444-22- but I forget if the last numbers are 5543 or 5544. Call me back !"""

In [ ]:
more_text_results = analyzer.analyze(text=more_text, 
                                    entities=supported_entities, 
                                    language='en')

In [ ]:
more_text_results

### "Anonymizing" <- more like pseudonymizing, Engine

Now that you've found the data you want to potentially remove or protect, you can use Presidio to replace, redact, hash, mask, encrypt. Check out the [documentation](https://microsoft.github.io/presidio/anonymizer/) for information on how to use each of the choices.

Please note: the "hash" method doesn't take any salt, so please use with care (or just avoid and use encrypt instead). 

In [ ]:
anonymizer = AnonymizerEngine()

anonymized_results = anonymizer.anonymize(
    text=text_to_pseudonymize,
    analyzer_results=analyzer_results,    
    operators={
        "DEFAULT": OperatorConfig("replace", 
                                  {"new_value": "<REDACTED>"}), 
        "PHONE_NUMBER": OperatorConfig("mask", {
                                        "type": "mask", 
                                        "masking_char" : "*",
                                        "chars_to_mask" : 12, 
                                        "from_end" : True}),
    }
)

In [ ]:
anonymized_results

## Try with your text!

- I've written out a way to generate an [AES key]() because that's the only method supported by the library.
- You can also test out [other methods in the documentation](https://microsoft.github.io/presidio/anonymizer/). If it has variables listed in the table, it means you need to supply those variables and their values in the function call.
- If you run into trouble, please raise your hand!

In [ ]:
def generate_aes_key(bit_length=256): 
    byte_length = int(bit_length / 8)
    return os.urandom(byte_length)

In [ ]:
key = generate_aes_key()

In [ ]:
key

In [ ]:
anonymized_results = anonymizer.anonymize(
    text=more_text,
    analyzer_results=more_text_results,    
    operators={
        "DEFAULT": OperatorConfig("encrypt", 
                                  {"key": key}), 
    }
)

In [ ]:
anonymized_results

In [ ]:
from presidio_anonymizer.operators import Decrypt

# Restore the original entity value
Decrypt().operate(text="DnPRRSR0S9DirDPcJCfEnfOE1YjayJhMOqor-Ea4Kr8=", params={"key": key})

## Shareout

- What do you like about the library? How could you see it helping your AI/ML workflows?
- What do you wish for that it doesn't do?
- How can we imagine supporting some of those wishes?